In [ ]:
%load_ext autoreload
%autoreload 2

from config import BYBIT_API_KEY, BYBIT_DEMO_API_KEY, BYBIT_SECRET_KEY, BYBIT_DEMO_SECRET_KEY
from config import OKX_DEMO_API_KEY, OKX_DEMO_SECRET_KEY, OKX_DEMO_PASSPHRASE, OKX_API_KEY, OKX_SECRET_KEY, OKX_PASSPHRASE
from config import GATE_DEMO_API_KEY, GATE_DEMO_SECRET_KEY
from config import host, user, password, db_name
from jaref_bot.db.db_manager import DBManager
from jaref_bot.data.http_api import ExchangeManager, BybitRestAPI, OKXRestAPI, GateIORestAPI

from jaref_bot.trading.trade_api import BybitTrade, OkxTrade, OkxClient, GateTrade
from jaref_bot.core.exceptions.database import OrderDuplicationError

import pandas as pd

from datetime import datetime, UTC
import requests
from dataclasses import dataclass
import hmac
import hashlib
import base64
import json
import random
import string
from time import sleep
from decimal import Decimal

db_params = {'host': host, 'user': user, 'password': password, 'database': db_name}
db_manager = DBManager(db_params)

In [ ]:
def get_position(demo, exc, symbol, order_type):
    exc_dict = {'bybit': BybitTrade, 'okx': OkxTrade, 'gate': GateTrade}
    exchange, market_type = exc.split('_')
    exc_manager = exc_dict[exchange](demo=demo)

    ct_val = coin_information[exc][symbol]['ct_val']

    pos = exc_manager.get_position(market_type='linear', symbol=symbol, order_type=order_type, ct_val=ct_val)
    if pos:
        db_manager.place_order(token=symbol, exchange=exchange, market_type=market_type, order_type=order_type, 
                               order_side=pos['order_side'], qty=pos['qty'], price=pos['price'], 
                               usdt_amount=pos['usdt_amount'], usdt_fee=pos['fee'], leverage=pos['leverage'])
        print('[ORDER CONFIRMED]')

In [ ]:
import pickle
with open("./data/coin_information.pkl", "rb") as f:
    coin_information = pickle.load(f)

In [ ]:
bybit_trade = BybitTrade(demo=True)
okx_trade = OkxTrade(demo=True)
gate_trade = GateTrade(demo=True)

In [ ]:
bybit_trade = BybitTrade(demo=False)
bybit_trade.set_leverage(market_type=market_type, symbol=symbol, lever=lever)


In [ ]:
# Открываем ордер
symbol = 'ADA_USDT'
market_type = 'linear'
order_type = 'market'
lever = 2
qty = 20

bybit_trade.set_leverage(market_type=market_type, symbol=symbol, lever=lever)
order_id = bybit_trade.market_order(market_type=market_type, symbol=symbol, side='buy', order_type=order_type, qty=qty)
get_position(demo=True, exc='bybit_linear', symbol=symbol, order_type=order_type)

In [ ]:
# Закрываем ордер
order = db_manager.get_order('ADA_USDT')
close_side = 'sell' if order['order_side'] == 'buy' else 'buy'

order_id = bybit_trade.market_order(market_type=market_type, symbol=symbol, side=close_side, order_type=order_type, qty=qty)
close_info = bybit_trade._get_order(market_type=market_type, order_id=order_id)

close_price = close_info['price']
close_usdt_amount = close_info['usdt_amount']
close_fee = close_info['fee']
close_price, close_usdt_amount, close_fee

db_manager.close_order(token=symbol, exchange='bybit', market_type=market_type, qty=qty,
                       close_price=close_price, close_usdt_amount=close_usdt_amount, close_fee=close_fee)

In [ ]:
def close_order(token):
    order = db_manager.get_order(token)
    close_side = 'sell' if order['order_side'] == 'buy' else 'buy'
    
    order_id = bybit_trade.market_order(market_type=market_type, symbol=symbol, side=close_side, order_type=order_type, qty=qty)
    close_info = bybit_trade._get_order(market_type=market_type, order_id=order_id) 
    
    close_price = close_info['price']
    close_usdt_amount = close_info['usdt_amount']
    close_fee = close_info['fee']
    close_price, close_usdt_amount, close_fee
    
    db_manager.close_order(token=symbol, exchange='bybit', market_type=market_type, qty=qty,
                           close_price=close_price, close_usdt_amount=close_usdt_amount, close_fee=close_fee)

In [ ]:
while True:
    current_data = db_manager.get_table('current_data')
    price = current_data[(current_data['token'] == symbol) & (current_data['exchange'] == 'bybit')]['bid_price'].item()

    if price < 0.638:
        close_order(symbol)
        break
    sleep(0.5)

#### exchanges api

In [ ]:
def get_missing_instr_info(exchange, market_type, symbol):
    exc_dict = {'bybit': BybitRestAPI, 'okx': OKXRestAPI, 'gate': GateIORestAPI}
    client = exc_dict[exchange](market_type)
    token = client._create_symbol_name(symbol)
    
    return client.get_instrument_data(symbol=token)

In [ ]:
def get_order_status(exchange, market_type, token):
    order_in_pending = db_manager.order_exists(table_name='pending_orders', exchange=exchange, market_type=market_type, token=token)
    order_in_current = db_manager.order_exists(table_name='current_orders', exchange=exchange, market_type=market_type, token=token)

    if not (order_in_pending or order_in_current):
        status = None
    elif order_in_pending and order_in_current:
        status = 'adding'
    elif order_in_pending and not order_in_current:
        status = 'placed'
    elif not order_in_pending and order_in_current:
        status = 'live'
    return status

In [ ]:
market_fees = {'bybit_spot': 0.001, 'bybit_linear': 0.00055, 'okx_spot': 0.001, 'okx_linear': 0.0005,
               'gate_spot': 0.002, 'gate_linear': 0.0005}

In [ ]:
# ====================================
# Инициация нужных криптобирж, рынков и БД
exc_manager = ExchangeManager()
exc_manager.add_market("bybit_linear", BybitRestAPI('linear'))
exc_manager.add_market("bybit_spot", BybitRestAPI('spot'))
exc_manager.add_market("okx_linear", OKXRestAPI('linear'))
exc_manager.add_market("okx_spot", OKXRestAPI('spot'))
exc_manager.add_market("gate_linear", GateIORestAPI('linear'))
exc_manager.add_market("gate_spot", GateIORestAPI('spot'))

db_params = {'host': host, 'user': user, 'password': password, 'database': db_name}
db_manager = DBManager(db_params)
print('auto copy from current_data to market_data', db_manager.get_auto_copy_trigger_state())

In [ ]:
coin_information = exc_manager.get_instrument_data()

In [ ]:
class OkxAccount(OkxClient):
    def __init__(self, demo=True, debug=True):
        OkxClient.__init__(self, demo=demo, debug=debug)

    # Get Balance
    def get_account_balance(self, symbol=''):
        params = {}
        if symbol:
            params['ccy'] = symbol
        return self._request_with_params(method='GET', request_path='/api/v5/account/balance', params=params)

    # Get Positions
    def get_positions(self, market_type='', symbol=''):
        if market_type == 'linear':
            market_type = 'SWAP'
        params = {'instType': market_type, 'instId': symbol}
        return self._request_with_params(method='GET', request_path='/api/v5/account/positions', params=params)

    def get_leverage(self, market_type, symbol, margin_mode):
        sym = self._create_symbol_name(market_type, symbol)
        params = {'instId': sym, 'mgnMode': margin_mode}
        return self._request_with_params('GET', request_path='/api/v5/account/leverage-info', params=params)
    
    def set_leverage(self, market_type, symbol, leverage, margin_mode, ccy='', pos_side=''):
        sym = self._create_symbol_name(market_type, symbol)
        params = {'lever': leverage, 'mgnMode': margin_mode, 'instId': sym, 'ccy': ccy, 'posSide': pos_side}
        return self._request_with_params('POST', request_path='/api/v5/account/set-leverage', params=params)

    def get_fee_rates(self, symbol, instId='', uly='', category='', instFamily=''):
        sym = self._create_symbol_name(market_type, symbol)
        params = {'instType': instType, 'instId': sym, 'uly': uly, 'category': category, 'instFamily': instFamily}
        return self._request_with_params(GET, FEE_RATES, params)

In [ ]:
account_client = OkxAccount(demo=True, debug=False)

In [ ]:
account_client.get_leverage(market_type='linear', symbol='1INCH', margin_mode='isolated')

In [ ]:
account_client.set_leverage(market_type='linear', symbol='BTC', leverage=1, margin_mode='isolated')

In [ ]:
# Закрытие позиции на спотовом рынке. Надо иметь в виду, что купленное кол-во токенов будет на размер fee меньше 
#   запрошенного. То есть при закрытии надо из qty вычитать fee, и это значение отправлять в заявку

# trade_client.market_order(market_type='spot', symbol='ADA', side='sell', order_type='market', qty=19.98)

#### current profit

In [ ]:
import pandas as pd

try:
    current_result_df = pd.read_parquet('./data/current_result_df.parquet')
    for token in current_result_df['token'].unique():
        if token == 'GMT_USDT':
            continue
        temp_df = current_result_df[current_result_df['token'] == token].copy()
    
        temp_df[['profit', 'diff']] = temp_df[['profit', 'diff']].astype(float)
        temp_df[['profit', 'diff']].plot(figsize=(12, 3), title=token);
except FileNotFoundError:
    print('Нет текущих ордеров')

In [ ]:
current_result_df

#### Поиск наилучших условий для входа и выхода

In [ ]:
%load_ext autoreload
%autoreload 2

from jaref_bot.db.db_manager import DBManager
import pandas as pd
from config import host, user, password, db_name
import matplotlib.pyplot as plt
from decimal import Decimal
from tqdm.notebook import tqdm

db_params = {'host': host, 'user': user, 'password': password, 'dbname': db_name}
db_manager = DBManager(db_params)

In [ ]:
market_fees = {'bybit_spot': 0.0018, 'bybit_linear': 0.001, 'okx_spot': 0.001, 'okx_linear': 0.0005,
               'gate_spot': 0.002, 'gate_linear': 0.0005}

In [ ]:
# db_manager.clear_old_data(8)

In [ ]:
# Загружаем таблицу и преобразуем данные в нужный формат
# df = db_manager.get_table('market_data_5s')
# df = df.sort_values(by='bucket').reset_index(drop=True)
# df[['avg_bid', 'avg_ask']] = df[['avg_bid', 'avg_ask']].astype(float)

In [ ]:
# Список нужных мне криптобирж
exchanges = ('bybit', 'okx', 'gate')

# Создаём пустой датафрейм для хранения результатов
stats_df = pd.DataFrame(columns = ['token', 'long_exc', 'short_exc', 'mean', 'std', 'max_diff'])

tokens = db_manager.get_unique_tokens()
for token in tqdm(tokens):
    tdf = db_manager.get_token_history(token)
    tdf = tdf.sort_values(by='bucket').reset_index(drop=True)
    tdf[['avg_bid', 'avg_ask']] = tdf[['avg_bid', 'avg_ask']].astype(float)

    for long_exc in exchanges:
        for short_exc in exchanges:
            lm_ask_price = pd.DataFrame()
            sm_bid_price = pd.DataFrame()
            
            if long_exc == short_exc:
                continue
            
            long_mask = (tdf['exchange'] == long_exc) & (tdf['market_type'] == 'linear')
            short_mask = (tdf['exchange'] == short_exc) & (tdf['market_type'] == 'linear')
            
            lm_ask_price = tdf[long_mask][['bucket', 'exchange', 'market_type', 'avg_ask']]
            sm_bid_price = tdf[short_mask][['bucket', 'exchange', 'market_type', 'avg_bid']]
            temp_df = lm_ask_price.merge(sm_bid_price, on='bucket', suffixes=('_long', '_short'))
            temp_df['diff'] = (temp_df['avg_bid'] / temp_df['avg_ask'] - 1) * 100
    
            minv, maxv, meanv, stdv = temp_df['diff'].agg(['min', 'max', 'mean', 'std'])
            max_diff = maxv - minv
    
            stats_df.loc[len(stats_df)] = {'token': token, 'long_exc': long_exc, 
                                'short_exc': short_exc, 'mean': meanv, 'std': stdv, 'max_diff': max_diff}

stats_df.dropna(inplace=True)
stats_df.reset_index(drop=True, inplace=True)
db_manager.update_stats(stats_df)
# stats_df.to_parquet('./data/stats_df.parquet')

In [ ]:
stats_df.tail(2)

#### Графики

In [ ]:
token = 'AIOZ_USDT'
upper_coef = 2.5
lower_coef = 2.5

long_exc = 'bybit'
short_exc = 'gate'

tdf = db_manager.get_token_history(token)
tdf[['avg_bid', 'avg_ask']] = tdf[['avg_bid', 'avg_ask']].astype(float)
long_mask = (tdf['exchange'] == long_exc) & (tdf['market_type'] == 'linear')
short_mask = (tdf['exchange'] == short_exc) & (tdf['market_type'] == 'linear')

lm_ask_price = tdf[long_mask][['bucket', 'exchange', 'market_type', 'avg_ask']]
sm_bid_price = tdf[short_mask][['bucket', 'exchange', 'market_type', 'avg_bid']]
temp_df = lm_ask_price.merge(sm_bid_price, on='bucket', suffixes=('_long', '_short'))
temp_df['diff'] = (temp_df['avg_bid'] / temp_df['avg_ask'] - 1) * 100
curr_diff = temp_df['diff'].iloc[-1]

minv, maxv, meanv, stdv = temp_df['diff'].agg(['min', 'max', 'mean', 'std'])
max_diff = maxv - minv
dev = (curr_diff - meanv) / stdv

print(f'{meanv=:.3f}, {stdv=:.3f}, {curr_diff=:.3f}, {dev=:.3f}')
# temp_df['diff'][:].astype(float).plot(figsize=(14, 3))
plt.figure(figsize=(18, 4))
plt.plot(temp_df['bucket'], temp_df['diff']);
plt.axhline(y=meanv, color='black', linestyle='-') # mean
plt.axhline(y=meanv - lower_coef * stdv, color='g', linestyle='-') # low bound
plt.axhline(y=meanv + upper_coef * stdv, color='g', linestyle='-'); # high bound

In [ ]:
# Условия выхода
token = 'LRC_USDT'

# Определяем рынки, на которых открыт ордер
current_orders = db_manager.get_table('current_orders')
orders = current_orders[current_orders['token'] == token]

long_market = orders[orders['order_side'] == 'buy']
short_market = orders[orders['order_side'] == 'sell']

long_exc = long_market['exchange'].item()
short_exc = short_market['exchange'].item()
long_mt = long_market['market_type'].item()
short_mt = short_market['market_type'].item()

# Определяем цены на момент открытия и начальный diff
long_open_price = long_market['price'].item().normalize()
short_open_price =  short_market['price'].item().normalize()
open_diff = round((short_open_price / long_open_price - 1) * 100, 3)

long_exc, short_exc, long_mt, short_mt, open_diff

In [ ]:
tdf = db_manager.get_token_history(token)
tdf[['avg_bid', 'avg_ask']] = tdf[['avg_bid', 'avg_ask']].astype(float)

long_mask = (tdf['exchange'] == long_exc) & (tdf['market_type'] == long_mt)
short_mask = (tdf['exchange'] == short_exc) & (tdf['market_type'] == short_mt)
tdf = tdf[long_mask | short_mask]

long_mask = long_mask.reindex(tdf.index, fill_value=False)
short_mask = short_mask.reindex(tdf.index, fill_value=False)

lm_sell_price = tdf[long_mask][['bucket', 'avg_bid']]
sm_buy_price = tdf[short_mask][['bucket', 'avg_ask']]
close_df = lm_sell_price.merge(sm_buy_price)
close_df['diff'] = (close_df['avg_bid'] / close_df['avg_ask'] - 1) * 100

In [ ]:
minv, maxv, meanv, stdv = close_df['diff'].astype(float).agg(['min', 'max', 'mean', 'std'])

upper_coef = 2.0
lower_coef = 2.0

close_diff = close_df['diff'].iloc[-1]
deviation = (close_diff - meanv) / stdv
print(f'{meanv=:.3f}, {stdv=:.3f}, current diff: {close_diff:.3f}, {deviation=:.1f}')

close_df['diff'][:].plot(figsize=(14, 3))
plt.axhline(y=meanv, color='r', linestyle='-') # mean
plt.axhline(y=meanv - lower_coef * stdv, color='g', linestyle='-') # low bound
plt.axhline(y=meanv + upper_coef * stdv, color='g', linestyle='-'); # high bound

In [ ]:
# Профит
long_market_fee = long_market['usdt_fee'].iloc[0].normalize()
short_market_fee = short_market['usdt_fee'].iloc[0].normalize()
qty = short_market['qty'].iloc[0].normalize()

long_fee = Decimal(market_fees[long_exc + '_' + long_mt])
short_fee = Decimal(market_fees[short_exc + '_' + long_mt])

def get_curr_profit(row):
    long_market_profit = Decimal(row['avg_bid']) * (1 - long_fee) - long_open_price * (1 + long_fee)
    short_market_profit = short_open_price * (1 - short_fee) - Decimal(row['avg_ask']) * (1 + short_fee)
    profit = qty * (long_market_profit + short_market_profit)

    # print(f'{row['bucket']}: bid: {row['avg_bid']}; ask: {row['avg_ask']}; long: {qty * long_market_profit:.3f}; short: {qty * short_market_profit:.3f}; total: {profit:.3f}; fee: {4 * short_fee:.4f}')
    
    return round(profit, 4)

close_df['profit'] = close_df.apply(get_curr_profit, axis=1)

In [ ]:
close_df['profit'][-1000:].astype(float).plot(figsize=(14, 3));

In [ ]:
# bid: 622.55; ask: 622.45; long: -6.141; short: 7.005; total: 0.864; fee: 0.0022

In [ ]:
current_data = db_manager.get_table('current_data')

In [ ]:
current_data

In [ ]:
close_df

#### Отправка тестовых запросов к биржам через aiohttp

In [ ]:
import asyncio
import aiohttp
import nest_asyncio
nest_asyncio.apply()

async def fetch(session, url):
    async with session.get(url, params=params) as response:
        return await response.json()

bb_url = "https://api-demo.bybit.com"
okx_url = 'https://www.okx.com'
gate_url = 'https://api.gateio.ws'

endpoint = '/api/v5/market/books'

params = {'instId': 'BNB-USDT-SWAP'}

urls = [okx_url + endpoint]
async with aiohttp.ClientSession() as session:
    tasks = [fetch(session, url) for url in urls]
    results = await asyncio.gather(*tasks)
print(results)

In [ ]:
for i in results[0]['result']['list']:
    print(i['symbol'])